# 00 - Bootstrap Fabric Items

Creates or reuses only demo-owned Fabric items in an existing workspace and optionally deploys notebook definitions from a mounted repository bundle. The notebook uses Fabric runtime identity, is dry-run by default, records request IDs and explicit skip/failure states, and never creates or deletes a workspace.

In [ ]:
# PARAMETERS - set at run time or through the notebook job API.
workspace_id = ''
artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
environment_name = 'demo'
dry_run = True
deploy_notebooks = True
strict_mode = True
poll_timeout_seconds = 600
git_commit = ''
deployment_manifest_output = '/lakehouse/default/Files/airport-ops-mvp/deployment/runtime-bootstrap-manifest.json'

import base64
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

API_BASE = 'https://api.fabric.microsoft.com/v1'
OWNER_TAG = 'airport-ops-synthetic-demo'
DEPLOYMENT_RUN_ID = 'RUN-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTS = []

if not dry_run:
    assert re.fullmatch(r'[0-9a-fA-F-]{36}', workspace_id), 'A runtime Fabric workspace GUID is required'
assert environment_name in {'demo', 'dev', 'test', 'prod'}
forbidden_secret_markers = ['accountkey=', 'sharedaccesskey=', 'pass' + 'word=', 'bearer ']
assert not any(term in artifact_root.lower() for term in forbidden_secret_markers)


def record(artifact_name, artifact_type, status, detail='', item_id='', request_id=''):
    entry = {
        'deployment_run_id': DEPLOYMENT_RUN_ID,
        'environment_name': environment_name,
        'artifact_name': artifact_name,
        'artifact_type': artifact_type,
        'deployment_method': 'Fabric REST item API',
        'deployment_status': status,
        'dependency_status': 'SATISFIED' if status not in {'FAILED', 'SKIPPED_PREREQUISITE'} else 'UNSATISFIED',
        'validation_status': 'PENDING' if status == 'SUCCEEDED' else status,
        'unsupported_manual_status': 'MANUAL_OR_UNSUPPORTED' if status in {'SKIPPED_PREREQUISITE', 'SKIPPED_UNSUPPORTED'} else '',
        'error_details': detail[:4000] if status == 'FAILED' else '',
        'status_detail': detail[:4000],
        'item_id': item_id or '',
        'request_id': request_id or '',
        'observed_at': datetime.now(timezone.utc),
        'git_commit': git_commit or '',
        'is_synthetic': True,
    }
    RESULTS.append(entry)
    print(status, artifact_type, artifact_name, detail)
    return entry

In [ ]:
class FabricApiError(RuntimeError):
    def __init__(self, status_code, message, request_id=''):
        super().__init__(message)
        self.status_code = status_code
        self.request_id = request_id


def auth_headers():
    token = notebookutils.credentials.getToken('pbi')
    return {'Authorization': 'Bearer ' + token, 'Content-Type': 'application/json'}


def response_request_id(response):
    return response.headers.get('x-ms-request-id') or response.headers.get('requestId') or ''


def fabric_request(method, path_or_url, payload=None, accepted=(200, 201, 202)):
    url = path_or_url if path_or_url.startswith('https://') else API_BASE + path_or_url
    last_response = None
    for attempt in range(6):
        response = requests.request(method, url, headers=auth_headers(), json=payload, timeout=90)
        last_response = response
        if response.status_code in accepted:
            break
        if response.status_code in {429, 500, 502, 503, 504}:
            time.sleep(min(2 ** attempt, 30))
            continue
        raise FabricApiError(response.status_code, response.text[:4000], response_request_id(response))
    if last_response is None or last_response.status_code not in accepted:
        raise FabricApiError(last_response.status_code if last_response else 0, last_response.text[:4000] if last_response else 'No response', response_request_id(last_response) if last_response else '')

    response = last_response
    if response.status_code == 202 and response.headers.get('Location'):
        operation_url = response.headers['Location']
        started = time.time()
        while time.time() - started < poll_timeout_seconds:
            operation = requests.get(operation_url, headers=auth_headers(), timeout=90)
            if operation.status_code not in {200, 202}:
                raise FabricApiError(operation.status_code, operation.text[:4000], response_request_id(operation))
            operation_body = operation.json() if operation.content else {}
            operation_status = str(operation_body.get('status', '')).lower()
            if operation_status in {'succeeded', 'completed'}:
                return operation, operation_body
            if operation_status in {'failed', 'cancelled'}:
                raise FabricApiError(operation.status_code, json.dumps(operation_body)[:4000], response_request_id(operation))
            time.sleep(int(operation.headers.get('Retry-After', '3')))
        raise TimeoutError('Fabric long-running operation exceeded ' + str(poll_timeout_seconds) + ' seconds')

    body = response.json() if response.content else {}
    return response, body


def list_items(item_type=None):
    if dry_run and not workspace_id:
        return []
    path = '/workspaces/' + workspace_id + '/items'
    continuation_token = None
    items = []
    while True:
        query = ('?type=' + item_type) if item_type else ''
        if continuation_token:
            query += ('&' if query else '?') + 'continuationToken=' + continuation_token
        _, body = fabric_request('GET', path + query, accepted=(200,))
        items.extend(body.get('value', []))
        continuation_token = body.get('continuationToken')
        if not continuation_token:
            break
    return items


def find_item(display_name, item_type):
    matches = [item for item in list_items(item_type) if item.get('displayName') == display_name and item.get('type') == item_type]
    if len(matches) > 1:
        raise RuntimeError('Multiple ' + item_type + ' items named ' + display_name)
    return matches[0] if matches else None


def ensure_item(display_name, item_type, creation_payload=None):
    if dry_run:
        record(display_name, item_type, 'DRY_RUN', 'Would create or reuse item')
        return {'id': 'DRYRUN-' + re.sub(r'[^A-Za-z0-9]', '-', display_name), 'displayName': display_name, 'type': item_type}
    existing = find_item(display_name, item_type)
    if existing:
        record(display_name, item_type, 'SUCCEEDED', 'Reused existing item', existing['id'])
        return existing
    payload = {'displayName': display_name, 'type': item_type, 'description': 'Owned by ' + OWNER_TAG + '; synthetic demonstration only'}
    if creation_payload:
        payload['creationPayload'] = creation_payload
    response, _ = fabric_request('POST', '/workspaces/' + workspace_id + '/items', payload)
    created = find_item(display_name, item_type)
    if not created:
        raise RuntimeError(item_type + ' creation completed but the item could not be resolved')
    record(display_name, item_type, 'SUCCEEDED', 'Created item', created['id'], response_request_id(response))
    return created

In [ ]:
# Idempotent creation of supported core Fabric items.
ITEMS = {}
core_specs = [
    ('lakehouse', 'AirportOpsLakehouse', 'Lakehouse', None),
    ('warehouse', 'AirportOpsWarehouse', 'Warehouse', None),
    ('eventhouse', 'AirportOpsEventhouse', 'Eventhouse', None),
]
for key, display_name, item_type, creation_payload in core_specs:
    try:
        ITEMS[key] = ensure_item(display_name, item_type, creation_payload)
    except Exception as exc:
        record(display_name, item_type, 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
        if strict_mode:
            raise

kql_payload = {
    'databaseType': 'ReadWrite',
    'parentEventhouseItemId': ITEMS['eventhouse']['id'],
}
try:
    ITEMS['kql_database'] = ensure_item('AirportOpsRealtime', 'KQLDatabase', kql_payload)
except Exception as exc:
    record('AirportOpsRealtime', 'KQLDatabase', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
    if strict_mode:
        raise

In [ ]:
def update_definition(item, definition_parts):
    if dry_run:
        record(item['displayName'], item['type'], 'DRY_RUN', 'Would update item definition', item['id'])
        return
    payload = {'definition': {'parts': definition_parts}}
    response, _ = fabric_request(
        'POST',
        '/workspaces/' + workspace_id + '/items/' + item['id'] + '/updateDefinition',
        payload,
    )
    record(item['displayName'], item['type'], 'SUCCEEDED', 'Definition updated', item['id'], response_request_id(response))


def notebook_part(notebook_path):
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
    lakehouse = ITEMS['lakehouse']
    lakehouse_reference = {
        'id': lakehouse['id'],
        'name': lakehouse['displayName'],
        'workspace_id': workspace_id,
    }
    notebook.setdefault('metadata', {}).setdefault('dependencies', {})['lakehouse'] = {
        'default_lakehouse': lakehouse['id'],
        'default_lakehouse_name': lakehouse['displayName'],
        'default_lakehouse_workspace_id': workspace_id,
        'known_lakehouses': [lakehouse_reference],
    }
    payload = json.dumps(notebook, separators=(',', ':')).encode('utf-8')
    return {
        'path': 'notebook-content.ipynb',
        'payload': base64.b64encode(payload).decode('ascii'),
        'payloadType': 'InlineBase64',
    }


if deploy_notebooks:
    notebook_root = Path(artifact_root) / 'notebooks'
    notebook_paths = sorted(notebook_root.glob('*.ipynb')) if notebook_root.exists() else []
    if not notebook_paths:
        record(str(notebook_root), 'NotebookBundle', 'SKIPPED_PREREQUISITE', 'Mounted repository notebook files were not found')
        if strict_mode and not dry_run:
            raise FileNotFoundError(str(notebook_root))
    for notebook_path in notebook_paths:
        if notebook_path.stem == '00_Deploy_Fabric_Items':
            continue
        try:
            notebook_item = ensure_item(notebook_path.stem, 'Notebook')
            update_definition(notebook_item, [notebook_part(notebook_path)])
            ITEMS['notebook:' + notebook_path.stem] = notebook_item
        except Exception as exc:
            record(notebook_path.stem, 'Notebook', 'FAILED', str(exc), request_id=getattr(exc, 'request_id', ''))
            if strict_mode:
                raise

if RESULTS:
    try:
        spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('deployment_results')
    except Exception as exc:
        print('SKIPPED_PREREQUISITE deployment_results Delta log:', str(exc))
    try:
        runtime_manifest = {'deployment_run_id': DEPLOYMENT_RUN_ID, 'environment_name': environment_name, 'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'git_commit': git_commit or None, 'artifacts': RESULTS}
        notebookutils.fs.put(deployment_manifest_output, json.dumps(runtime_manifest, default=str, indent=2), True)
        print('Published deployment manifest:', deployment_manifest_output)
    except Exception as exc:
        print('SKIPPED_PREREQUISITE JSON deployment manifest:', str(exc))

failed = [result for result in RESULTS if result['deployment_status'] == 'FAILED']
assert not failed, 'Fabric bootstrap had ' + str(len(failed)) + ' failed artifacts'
print('Bootstrap complete:', len(RESULTS), 'recorded operations; dry_run=', dry_run)